In [2]:
# ============================================
# NOTEBOOK 3: INSTRUCTOR PERFORMANCE ANALYSIS
# ============================================

# %% Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# %% Load Integrated Data
df = pd.read_csv('../data/integrated_data.csv')
print(f"✅ Loaded integrated data: {df.shape}")

# %% Instructor-Level Aggregation
print("\n" + "=" * 60)
print("INSTRUCTOR-LEVEL METRICS")
print("=" * 60)

instructor_metrics = df.groupby('TeacherID').agg({
    'TeacherName': 'first',
    'Age': 'first',
    'Gender': 'first',
    'Expertise': 'first',
    'YearsOfExperience': 'first',
    'TeacherRating': 'first',
    'CourseID': 'nunique',  # Number of unique courses taught
    'TransactionID': 'count',  # Total enrollments
    'CourseRating': ['mean', 'std', 'min', 'max']
}).reset_index()

# Flatten column names
instructor_metrics.columns = [
    'TeacherID', 'TeacherName', 'Age', 'Gender', 'Expertise',
    'YearsOfExperience', 'TeacherRating', 'CoursesTaught',
    'TotalEnrollments', 'AvgCourseRating', 'CourseRatingStd',
    'MinCourseRating', 'MaxCourseRating'
]

# Calculate Rating Consistency Index
instructor_metrics['RatingConsistency'] = (
    1 - (instructor_metrics['CourseRatingStd'] / instructor_metrics['AvgCourseRating'])
).fillna(1).clip(0, 1)

# Calculate Rating Gap
instructor_metrics['RatingGap'] = (
    instructor_metrics['TeacherRating'] - instructor_metrics['AvgCourseRating']
)

print(f"\n✅ Instructor metrics calculated for {len(instructor_metrics)} instructors")
print("\n", instructor_metrics.head())

# Save instructor metrics
instructor_metrics.to_csv('../outputs/tables/instructor_metrics.csv', index=False)

# %% Top Performers
print("\n" + "=" * 60)
print("TOP PERFORMING INSTRUCTORS")
print("=" * 60)

top_20 = instructor_metrics.nlargest(20, 'TeacherRating')[
    ['TeacherName', 'Expertise', 'YearsOfExperience', 'TeacherRating', 
     'AvgCourseRating', 'TotalEnrollments', 'RatingConsistency']
]

print("\n🏆 Top 20 Instructors by Rating:")
print(top_20.to_string(index=False))

# Visualize Top 20
plt.figure(figsize=(14, 8))
plt.barh(range(20), top_20['TeacherRating'], color='gold', edgecolor='black')
plt.yticks(range(20), top_20['TeacherName'])
plt.xlabel('Teacher Rating', fontsize=12)
plt.title('Top 20 Instructors by Rating', fontsize=16, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/top_20_instructors.png', dpi=300, bbox_inches='tight')
plt.show()

# %% Bottom Performers
print("\n" + "=" * 60)
print("INSTRUCTORS NEEDING IMPROVEMENT")
print("=" * 60)

bottom_20 = instructor_metrics.nsmallest(20, 'TeacherRating')[
    ['TeacherName', 'Expertise', 'YearsOfExperience', 'TeacherRating', 
     'AvgCourseRating', 'TotalEnrollments']
]

print("\n⚠️ Bottom 20 Instructors:")
print(bottom_20.to_string(index=False))

# %% Performance Distribution by Experience
print("\n" + "=" * 60)
print("PERFORMANCE BY EXPERIENCE LEVEL")
print("=" * 60)

# Create experience categories for instructor metrics
instructor_metrics['ExperienceCategory'] = pd.cut(
    instructor_metrics['YearsOfExperience'],
    bins=[0, 3, 7, 15, 100],
    labels=['Beginner', 'Intermediate', 'Experienced', 'Veteran']
)

exp_performance = instructor_metrics.groupby('ExperienceCategory').agg({
    'TeacherRating': ['mean', 'median', 'std'],
    'AvgCourseRating': 'mean',
    'TotalEnrollments': 'sum',
    'TeacherID': 'count'
}).round(2)

print("\n📊 Performance by Experience Category:")
print(exp_performance)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Average ratings by experience
exp_avg = instructor_metrics.groupby('ExperienceCategory')[
    ['TeacherRating', 'AvgCourseRating']
].mean()

exp_avg.plot(kind='bar', ax=axes[0], color=['#FF6B6B', '#4ECDC4'])
axes[0].set_title('Average Ratings by Experience Level', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Rating')
axes[0].set_xlabel('Experience Category')
axes[0].legend(['Teacher Rating', 'Course Rating'])
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Total enrollments by experience
exp_enrollments = instructor_metrics.groupby('ExperienceCategory')['TotalEnrollments'].sum()
axes[1].bar(exp_enrollments.index, exp_enrollments.values, color='mediumpurple')
axes[1].set_title('Total Enrollments by Experience Level', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Total Enrollments')
axes[1].set_xlabel('Experience Category')
axes[1].grid(axis='y', alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/figures/performance_by_experience.png', dpi=300, bbox_inches='tight')
plt.show()

# %% Performance by Gender
print("\n" + "=" * 60)
print("PERFORMANCE BY GENDER")
print("=" * 60)

gender_performance = instructor_metrics.groupby('Gender').agg({
    'TeacherRating': ['mean', 'median', 'std'],
    'AvgCourseRating': 'mean',
    'TotalEnrollments': ['sum', 'mean'],
    'TeacherID': 'count'
}).round(2)

print("\n📊 Performance by Gender:")
print(gender_performance)

# Statistical test
male_ratings = instructor_metrics[instructor_metrics['Gender'] == 'Male']['TeacherRating']
female_ratings = instructor_metrics[instructor_metrics['Gender'] == 'Female']['TeacherRating']

t_stat, p_value = stats.ttest_ind(male_ratings, female_ratings)
print(f"\n🔬 T-Test Results:")
print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value: {p_value:.4f}")

if p_value < 0.05:
    print("✅ Significant difference in ratings between genders")
else:
    print("❌ No significant difference in ratings between genders")

# %% Performance by Expertise
print("\n" + "=" * 60)
print("PERFORMANCE BY EXPERTISE AREA")
print("=" * 60)

expertise_performance = instructor_metrics.groupby('Expertise').agg({
    'TeacherRating': ['mean', 'std'],
    'AvgCourseRating': 'mean',
    'TotalEnrollments': 'sum',
    'TeacherID': 'count',
    'RatingConsistency': 'mean'
}).round(2)

expertise_performance.columns = [
    'AvgTeacherRating', 'StdTeacherRating',
    'AvgCourseRating', 'TotalEnrollments',
    'NumInstructors', 'AvgConsistency'
]

print("\n📊 Performance by Expertise:")
print(expertise_performance.sort_values('AvgTeacherRating', ascending=False))

# Save
expertise_performance.to_csv('../outputs/tables/expertise_performance.csv')

# Visualize
plt.figure(figsize=(14, 8))
expertise_sorted = expertise_performance.sort_values('AvgTeacherRating', ascending=True)
plt.barh(range(len(expertise_sorted)), expertise_sorted['AvgTeacherRating'], 
         color='teal', edgecolor='black')
plt.yticks(range(len(expertise_sorted)), expertise_sorted.index)
plt.xlabel('Average Teacher Rating', fontsize=12)
plt.title('Average Teacher Rating by Expertise Area', fontsize=16, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.axvline(instructor_metrics['TeacherRating'].mean(), color='red', 
            linestyle='--', label='Overall Average')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/performance_by_expertise.png', dpi=300, bbox_inches='tight')
plt.show()

# %% Rating Consistency Analysis
print("\n" + "=" * 60)
print("RATING CONSISTENCY ANALYSIS")
print("=" * 60)

# Most consistent instructors
most_consistent = instructor_metrics.nlargest(10, 'RatingConsistency')[
    ['TeacherName', 'Expertise', 'TeacherRating', 'RatingConsistency', 
     'CoursesTaught', 'CourseRatingStd']
]

print("\n✅ Most Consistent Instructors:")
print(most_consistent.to_string(index=False))

# Least consistent
least_consistent = instructor_metrics.nsmallest(10, 'RatingConsistency')[
    ['TeacherName', 'Expertise', 'TeacherRating', 'RatingConsistency', 
     'CoursesTaught', 'CourseRatingStd']
]

print("\n⚠️ Least Consistent Instructors:")
print(least_consistent.to_string(index=False))

# %% Scatter: Consistency vs Rating
plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    instructor_metrics['RatingConsistency'],
    instructor_metrics['TeacherRating'],
    c=instructor_metrics['TotalEnrollments'],
    s=instructor_metrics['CoursesTaught'] * 10,
    alpha=0.6,
    cmap='viridis'
)
plt.colorbar(scatter, label='Total Enrollments')
plt.xlabel('Rating Consistency Index', fontsize=12)
plt.ylabel('Teacher Rating', fontsize=12)
plt.title('Teacher Rating vs Consistency (size = courses taught)', 
          fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/consistency_vs_rating.png', dpi=300, bbox_inches='tight')
plt.show()

# %% [markdown]
# ## ✅ Key Findings
# 
# 1. Top Performers: Identified 20 highest-rated instructors
# 2. Experience Impact: [Your findings]
# 3. Consistency: [Your findings]
# 4. Expertise Gaps: [Your findings]

print("\n" + "=" * 60)
print("✅ NOTEBOOK 3 COMPLETED SUCCESSFULLY!")
print("=" * 60)


FileNotFoundError: [Errno 2] No such file or directory: '../data/integrated_data.csv'